# 09 Published baseline under one protocol

**Status: NOT RUN ON COLAB.** Generated by `scripts/make_notebooks.py`. Any output cells in the repository copy come from a local CPU smoke run (`SMOKE=True`, 20 samples), never from a real run.

| | |
|---|---|
| Purpose | (1) Implement the metric definitions of Nguyen et al. (arXiv:2401.10337, App. D.1) exactly and score OUR systems with them, so that R@1 and F1@3 are compared like with like. (2) Document why their model was not re-run and give the commands to do it. |
| Inputs | saved scores; (optional) a clone of `github.com/tumeteor/mitre-ttp-mapping` |
| Outputs | `baseline_protocol.json` |
| Runtime (T4 GPU, estimate) | 1 min for part 1 (CPU); part 2 about 24 h per run on a V100 according to the paper |
| Fills in `thesis_numbers.json` | T5.baseline_same_protocol |

Run the cells top to bottom. Set `SMOKE = True` in the setup cell for a dry run on 20 samples. Every run writes a manifest with input and output hashes; the last cell prints an evidence table read from the saved files.

The paper's numbers are NOT typed into this notebook. They are read from `run_match_1/final_results_table.csv` (the row the original notebook copied from the paper) so that the comparison table has a traceable source.

In [ ]:
NOTEBOOK = "09_baseline_same_protocol"
NEEDS_GPU = False      # True: the notebook stops at once if Colab gave no GPU

# ---- setup (same in every notebook) ----
import os, sys, json, ast, time, hashlib, random, platform, subprocess, re
from pathlib import Path
import numpy as np, pandas as pd

SMOKE = os.environ.get("TTPMAP_SMOKE", "0") == "1"      # set SMOKE = True by hand for a 20-sample dry run
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount failed:", type(e).__name__, e)
    if not os.path.isdir("/content/drive/MyDrive") and not SMOKE:
        raise RuntimeError("Google Drive is not mounted. The Colab extension for VS Code cannot mount Drive (known limitation of the extension), "
                           "and these notebooks read their inputs from and save their results to Drive. Open this notebook in the browser instead: "
                           "https://colab.research.google.com > File > Upload notebook (or open it from Drive). See notebooks/COLAB_RUNBOOK.md, section 0.")
BASE = Path("/content/drive/MyDrive/KDU/Thesis") if IN_COLAB else Path(os.environ.get("TTPMAP_ROOT", r"E:\Osamas Paper"))
DATA_DIR, CLS_ROOT, MATCH_ROOT, SHAP_DIR = BASE / "Dataset", BASE / "Exp_1_4_run_tram_2_epoc_30", BASE / "run_match_1", BASE / "Exp_3_0_shap"
MATCH2 = BASE / "run_match_2_saved"                      # NEW folder: matcher retrained WITH saved weights (notebook 04)
OUT = (Path(os.environ["TTPMAP_SMOKE_OUT"]) if os.environ.get("TTPMAP_SMOKE_OUT") else BASE / "showcase_runs") / NOTEBOOK
OUT.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 43, 44)
if IN_COLAB and not SMOKE and not (BASE / "Dataset" / "tram_test.tsv").exists():
    _msg = (f"Project files not found under {BASE}. Upload the folder E:\\Osamas Paper\\_showcase\\drive_upload\\KDU into 'My Drive' "
            "(drive.google.com, same Google account as this Colab session), then run this cell again. See notebooks/COLAB_RUNBOOK.md, section 0.")
    if NOTEBOOK == "00_setup_and_verify": print("WARNING:", _msg)
    else: raise RuntimeError(_msg)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def env_info():
    info = {"python": platform.python_version(), "platform": platform.platform(), "in_colab": IN_COLAB, "smoke": SMOKE}
    try:
        import torch
        info.update(torch=torch.__version__, cuda=torch.cuda.is_available(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
    except ImportError:
        pass
    return info

def write_manifest(run_id, system, seed, config, inputs, outputs, metrics, status):
    """One manifest per run: what went in, what came out, with hashes. Read later by the ingest step."""
    m = {"run_id": run_id, "system": system, "seed": seed, "config_hash": hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16], "config": config,
         "env": {k: str(v) for k, v in env_info().items()}, "input_sha256": {str(p): sha256(p) for p in inputs if Path(p).exists()},
         "output_sha256": {str(p): sha256(p) for p in outputs if Path(p).exists()}, "metrics": {k: float(v) for k, v in metrics.items()},
         "status": "SMOKE" if SMOKE else status, "written": time.strftime("%Y-%m-%dT%H:%M:%S")}
    (OUT / f"manifest_{run_id}.json").write_text(json.dumps(m, indent=1)); return m

print(f"notebook={NOTEBOOK} | SMOKE={SMOKE} | IN_COLAB={IN_COLAB} | BASE={BASE} | OUT={OUT}")
print(env_info())
import importlib.metadata as _md
print("packages:", {p: (_md.version(p) if p in {d.metadata["Name"].lower() for d in _md.distributions() if d.metadata["Name"]} else "not installed")
                    for p in ("torch", "transformers", "scikit-learn", "numpy", "pandas", "shap", "scipy")})
_gpu = env_info().get("cuda", False)
if NEEDS_GPU and not _gpu and not SMOKE:
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU > Save, then run this cell again. (Set SMOKE = True only for a dry run.)")
if not NEEDS_GPU and not _gpu:
    print("note: no GPU; this notebook runs on CPU in a few minutes")

In [ ]:
# ---- data, label space, gold matrices (same as Exp_1_4 CELL 4 and Exp_2.1 CELL 2) ----
def _labels(x): return sorted({str(t).strip().upper() for t in ast.literal_eval(x)})
def load_split(name):
    p = DATA_DIR / f"tram_{name}.tsv"
    if p.exists():
        df = pd.read_csv(p, sep="\t").rename(columns={"text1": "text"})
    elif name == "train":                                   # local fallback: the tram slice of combined_train.csv is the same data
        df = pd.read_csv(DATA_DIR / "combined_train.csv"); df = df[df["source_dataset"] == "tram"].rename(columns={"text1": "text"})
    else:
        raise FileNotFoundError(p)
    df = df[["text", "labels"]].copy(); df["labels"] = df["labels"].apply(_labels)
    return df.drop_duplicates(subset=["text"]).reset_index(drop=True) if name == "train" else df.reset_index(drop=True)

train, dev, test = (load_split(s) for s in ("train", "dev", "test"))
desc = pd.read_csv(DATA_DIR / "ttp-desc-mappings.csv", sep="\t")
desc["ttp"] = desc["ttp"].str.lower().str.strip()
desc["description"] = desc["description"].fillna("").str.replace(r"\s+", " ", regex=True).str.replace(r"\bcitation:\s*", "", regex=True).str.strip()
desc = desc.drop_duplicates("ttp"); desc = desc[~desc["ttp"].isin({"t1060", "t1107", "t1151", "t1188"})].reset_index(drop=True)
LABELS = [t.upper() for t in desc["ttp"]]; LABEL_TEXTS = desc["description"].tolist(); L = len(LABELS); lab2i = {l: i for i, l in enumerate(LABELS)}
def to_Y(df):
    Y = np.zeros((len(df), L), dtype=bool)
    for i, ys in enumerate(df["labels"]):
        for y in ys: Y[i, lab2i[y]] = True
    return Y
Y_train, Y_dev, Y_test = to_Y(train), to_Y(dev), to_Y(test)
train_count = Y_train.sum(0); SEEN = train_count > 0; seen_idx = np.where(SEEN)[0]
assert (len(train), len(dev), len(test), L, int(SEEN.sum())) == (3468, 604, 725, 733, 180), "unexpected data shape"
TEST_IDS = [f"test-{i:04d}" for i in range(len(test))]; DEV_IDS = [f"dev-{i:04d}" for i in range(len(dev))]
N_SMOKE = 20
print(f"train={len(train)} dev={len(dev)} test={len(test)} labels={L} seen={int(SEEN.sum())}")

In [ ]:
# ---- library cell: generated from src/ttpmap (metrics.py, fusion.py, stats.py); do not edit here ----
import itertools, math
from functools import reduce
from typing import Dict, Iterable, List, Sequence, Tuple, Literal
import numpy as np
from scipy.stats import binomtest
F1Variant = Literal['notebook', 'seen_test', 'all_test']
ORDER = ('matcher', 'svm', 'roberta', 'secbert')
BOOT_N, BOOT_SEED = 10_000, 20261114

def rank_topk(scores: np.ndarray, k: int) -> np.ndarray:
    """Indices of the k best labels per row; deterministic under ties."""
    return np.argsort(-scores, axis=1, kind="stable")[:, :k]


def hit_at_1(scores: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """Per-sentence 0/1 vector: is the top-1 label a gold label? Mean of it = P@1."""
    top = rank_topk(scores, 1)[:, 0]
    return Y[np.arange(len(Y)), top].astype(float)


def precision_at_k(scores: np.ndarray, Y: np.ndarray, k: int) -> np.ndarray:
    """Per-sentence (number of gold labels in the top k) / k."""
    return np.take_along_axis(Y, rank_topk(scores, k), axis=1).sum(1) / float(k)


def rr_at_k(scores: np.ndarray, Y: np.ndarray, k: int = 3) -> np.ndarray:
    """Per-sentence reciprocal rank of the first gold label inside the top k; 0 if none. Mean = MRR@k."""
    hits = np.take_along_axis(Y, rank_topk(scores, k), axis=1)
    first = hits.argmax(1) + 1
    return np.where(hits.any(1), 1.0 / first, 0.0)


def recall_at_k_micro(scores: np.ndarray, Y: np.ndarray, k: int) -> float:
    """Gold (sentence, label) pairs found in the top k / all gold pairs (the notebooks' "R@k")."""
    return float(np.take_along_axis(Y, rank_topk(scores, k), axis=1).sum() / Y.sum())


def recall_at_k_per_sample(scores: np.ndarray, Y: np.ndarray, k: int) -> float:
    """Mean over sentences of hits/|gold| (the definition printed in Nguyen et al., App. D.1)."""
    hits = np.take_along_axis(Y, rank_topk(scores, k), axis=1).sum(1)
    return float((hits / np.maximum(Y.sum(1), 1)).mean())


def apply_threshold(scores: np.ndarray, thr: float) -> np.ndarray:
    """Multi-label decision: score >= thr; a row with no label gets its arg-max label (never predict nothing)."""
    P = scores >= thr
    empty = P.sum(1) == 0
    P[np.where(empty)[0], scores[empty].argmax(1)] = True
    return P


def _prf(P: np.ndarray, Y: np.ndarray) -> Dict[str, float]:
    tp = int((P & Y).sum()); fp = int((P & ~Y).sum()); fn = int((~P & Y).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0}


def f1_per_label(P: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """Label-wise F1 with zero_division = 0."""
    tp = (P & Y).sum(0).astype(float); fp = (P & ~Y).sum(0); fn = (~P & Y).sum(0)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)


def f1_scores(P: np.ndarray, Y: np.ndarray, seen: np.ndarray, variant: F1Variant) -> Dict[str, float]:
    """Micro and macro F1 in one of three explicit variants.

    P, Y: bool (n, n_labels) in the FULL label space; ``seen``: bool (n_labels,) = label occurs in training.

    * ``notebook``  : only the columns of labels seen in training are scored. Gold labels never seen in training are
      ignored (not counted as misses). Macro over seen labels that occur in this split. This is what the original
      notebooks report.
    * ``seen_test`` : micro counts unseen gold labels as false negatives; macro over seen labels that occur in this split.
    * ``all_test``  : micro as in ``seen_test``; macro over ALL labels that occur in this split (unseen labels score 0).
    """
    present = Y.sum(0) > 0
    f = f1_per_label(P, Y)
    if variant == "notebook":
        m = _prf(P[:, seen], Y[:, seen]); macro = f[present & seen].mean()
    elif variant == "seen_test":
        m = _prf(P, Y); macro = f[present & seen].mean()
    elif variant == "all_test":
        m = _prf(P, Y); macro = f[present].mean()
    else:
        raise ValueError(variant)
    return {"micro_f1": m["f1"], "micro_precision": m["precision"], "micro_recall": m["recall"], "macro_f1": float(macro),
            "n_macro_labels": int((present & seen).sum() if variant != "all_test" else present.sum())}


class LeakageError(RuntimeError):
    """Raised when a tuning function is handed anything other than the dev split."""


def to_full(scores: np.ndarray, cols: np.ndarray, L: int, fill: float = np.nan) -> np.ndarray:
    F = np.full((scores.shape[0], L), fill); F[:, cols] = scores
    return F


def zrow(F: np.ndarray) -> np.ndarray:
    """Per-row z-score over the non-NaN labels; NaN (unknown label) -> 0."""
    m = np.nanmean(F, axis=1, keepdims=True); s = np.nanstd(F, axis=1, keepdims=True) + 1e-9
    return np.nan_to_num((F - m) / s, nan=0.0)


def fuse(Z: Dict[str, np.ndarray], w: Sequence[float]) -> np.ndarray:
    return sum(wi * Z[n] for wi, n in zip(w, ORDER))


def coarse_grid() -> List[Tuple[int, ...]]:
    """{0,1,2,3}^4 without the all-zero setting: 255 settings (Exp_2.1 CELL 8)."""
    return [w for w in itertools.product([0, 1, 2, 3], repeat=4) if sum(w)]


def wide_grid() -> List[Tuple[int, ...]]:
    """{0,1,2,3,4,6,8}^4 reduced by gcd: 1,711 distinct settings (Exp_2.1 CELL 9)."""
    seen, out = set(), []
    for w in itertools.product([0, 1, 2, 3, 4, 6, 8], repeat=4):
        g = reduce(math.gcd, w)
        if g == 0:
            continue
        w = tuple(x // g for x in w)
        if w not in seen:
            seen.add(w); out.append(w)
    return out


def select_weights(Z_dev: Dict[str, np.ndarray], Y_dev: np.ndarray, grid: Iterable[Tuple[int, ...]], *, split: str,
                   sentence_ids: Sequence[str] | None = None, forbidden_ids: Sequence[str] = ()) -> Tuple[Tuple[int, ...], Tuple[float, float]]:
    """Pick fusion weights. Objective: DEV MRR@3 (4 d.p.), tie-break DEV P@1; first strictly better setting wins.

    Guard: ``split`` must be the string "dev", and if ``sentence_ids`` are given none may be in ``forbidden_ids``
    (pass the test-set ids there). Violations raise LeakageError.
    """
    if split != "dev":
        raise LeakageError(f"weights may be tuned on dev only, got split={split!r}")
    if sentence_ids is not None and set(sentence_ids) & set(forbidden_ids):
        raise LeakageError("test sentence ids reached a tuning function")
    best = None
    for w in grid:
        Sc = fuse(Z_dev, w)
        key = (round(float(rr_at_k(Sc, Y_dev, 3).mean()), 4), round(float(hit_at_1(Sc, Y_dev).mean()), 4))
        if best is None or key > best[0]:
            best = (key, tuple(w))
    return best[1], best[0]


def select_threshold(scores_dev: np.ndarray, Y_dev: np.ndarray, grid: Sequence[float], *, split: str) -> float:
    """Global decision threshold maximising DEV micro-F1 over the model's own label columns (first maximum wins)."""
    if split != "dev":
        raise LeakageError(f"thresholds may be tuned on dev only, got split={split!r}")
    def mf(t):
        P = scores_dev >= t
        tp = (P & Y_dev).sum(); d = 2 * tp + (P & ~Y_dev).sum() + (~P & Y_dev).sum()
        return 2.0 * tp / d if d else 0.0
    return float(grid[int(np.argmax([mf(t) for t in grid]))])


def bootstrap_ci(x: np.ndarray, n_boot: int = BOOT_N, seed: int = BOOT_SEED, level: float = 0.95) -> Dict[str, float]:
    """Percentile CI of the mean of a per-sentence vector (unit of resampling = sentence)."""
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return {"mean": np.nan, "lo": np.nan, "hi": np.nan, "n": 0}
    rng = np.random.default_rng(seed)
    means = x[rng.integers(0, len(x), (n_boot, len(x)))].mean(1)
    a = (1 - level) / 2
    return {"mean": float(x.mean()), "lo": float(np.quantile(means, a)), "hi": float(np.quantile(means, 1 - a)), "n": int(len(x))}


def paired_bootstrap(a: np.ndarray, b: np.ndarray, n_boot: int = BOOT_N, seed: int = BOOT_SEED) -> Dict[str, float]:
    """Paired bootstrap of mean(a - b) with a percentile 95% CI and a two-sided bootstrap p-value."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    bs = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    return {"delta": float(d.mean()), "lo": float(np.quantile(bs, 0.025)), "hi": float(np.quantile(bs, 0.975)),
            "p_boot": float(min(1.0, 2 * min((bs <= 0).mean(), (bs >= 0).mean())))}


def mcnemar_exact(a: np.ndarray, b: np.ndarray) -> Dict[str, float]:
    """Exact two-sided McNemar on two 0/1 hit vectors. b_ = a right & b wrong, c_ = a wrong & b right."""
    a = np.asarray(a).astype(bool); b = np.asarray(b).astype(bool)
    b_ = int((a & ~b).sum()); c_ = int((~a & b).sum())
    p = binomtest(b_, b_ + c_, 0.5).pvalue if b_ + c_ else 1.0
    return {"b": b_, "c": c_, "p": float(p)}


def holm(pvalues: Sequence[float]) -> List[float]:
    """Holm step-down adjusted p-values, returned in the input order."""
    p = np.asarray(pvalues, dtype=float); m = len(p)
    order = np.argsort(p); adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1.0, running)
    return adj.tolist()

In [ ]:
# ---- saved model outputs + deterministic SVM refit ----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train["text"])
svm = OneVsRestClassifier(LinearSVC(C=1.0, random_state=42)).fit(vec.transform(train["text"]), Y_train[:, seen_idx].astype(int))
svm_raw = {"dev": svm.decision_function(vec.transform(dev["text"])), "test": svm.decision_function(vec.transform(test["text"]))}
cls_raw, cls_cols, cls_thr = {}, {}, {}
for tag in ("roberta", "secbert"):
    for s in SEEDS:
        p = CLS_ROOT / f"{tag}_pw_s{s}"
        cls_cols[tag, s] = np.array([lab2i[str(c).upper()] for c in np.load(p / "mlb_classes.npy", allow_pickle=True)])
        cls_raw[tag, s] = {"dev": np.load(p / "dev_probs.npy"), "test": np.load(p / "test_probs.npy")}
        cls_thr[tag, s] = json.load(open(p / "metrics.json"))["threshold"]
def matcher_dir(s):
    """Prefer the retrained matcher (with saved weights) when it exists; otherwise the original score files."""
    d = MATCH2 / f"ft20_s{s}"
    return d if (d / "test_scores.npy").exists() else MATCH_ROOT / f"ft20_s{s}"
m_raw = {s: {sp: np.load(matcher_dir(s) / f"{sp}_scores.npy") for sp in ("dev", "test")} for s in SEEDS}
MATCHER_SOURCE = {s: str(matcher_dir(s).parent.name) for s in SEEDS}
def components(split, mode="avg", seed=None):
    Z = {"svm": zrow(to_full(svm_raw[split], seen_idx, L))}
    for tag in ("roberta", "secbert"):
        Z[tag] = zrow(to_full(cls_raw[tag, seed][split], cls_cols[tag, seed], L)) if mode == "paired" else zrow(np.mean([to_full(cls_raw[tag, s][split], cls_cols[tag, s], L) for s in SEEDS], axis=0))
    Z["matcher"] = zrow(np.mean([m_raw[s][split] for s in SEEDS], axis=0)) if mode == "avg" else zrow(m_raw[seed][split])
    return Z
print("matcher scores from:", MATCHER_SOURCE)

In [ ]:
# Metric definitions exactly as printed in the paper (App. D.1):
#   P@k   = (1/k) * sum_{i<=k} 1[l_i in gold]                     per sample, then averaged over samples
#   R@k   = (1/|Q|) * sum_{i<=k} 1[l_i in gold], |Q| = number of gold labels of the sample; averaged over samples
#   F1@k  = 2 P@k R@k / (P@k + R@k)                               computed from the averaged P@k and R@k
#   MRR@k = (1/S) * sum_i 1/rank_i, rank of the first gold label inside the top k (a sample without a hit contributes 0: not stated in the paper, assumed)
def paper_metrics(S, Y):
    out = {}
    for k in (1, 3):
        hits = np.take_along_axis(Y, rank_topk(S, k), axis=1).sum(1); p = float((hits / k).mean()); r = float((hits / np.maximum(Y.sum(1), 1)).mean())
        out[f"P@{k}"] = p; out[f"R@{k}"] = r; out[f"F1@{k}"] = 2 * p * r / (p + r) if p + r else 0.0
    out["MRR@3"] = float(rr_at_k(S, Y, 3).mean()); out["R@1_micro_as_in_our_notebooks"] = recall_at_k_micro(S, Y, 1)
    return out
par = np.array([l.split(".")[0] for l in LABELS]); up = sorted(set(par)); PM = np.zeros((L, len(up))); PM[np.arange(L), [up.index(p) for p in par]] = 1
def technique_level(S, Y):                                                 # "resolve sub-techniques to their super-techniques" (paper Table 3)
    St = np.full((len(S), len(up)), -np.inf)
    for j in range(L): St[:, up.index(par[j])] = np.maximum(St[:, up.index(par[j])], S[:, j])
    return paper_metrics(St, (Y @ PM) > 0)
Za = components("test", "avg"); n_ = N_SMOKE if SMOKE else len(test)
systems = {"svm": to_full(svm_raw["test"], seen_idx, L, -1e9), "matcher_ensemble": Za["matcher"], "fusion_3111": fuse(Za, (3, 1, 1, 1))}
res = {}
for nm, S in systems.items():
    for k, v in paper_metrics(S[:n_], Y_test[:n_]).items(): res[f"{nm}_{k}"] = v
    for k, v in technique_level(S[:n_], Y_test[:n_]).items(): res[f"{nm}_techlevel_{k}"] = v
ref = {}
p = MATCH_ROOT / "final_results_table.csv"
if p.exists():
    fr = pd.read_csv(p); row = fr[fr["model"].str.startswith("Paper")].iloc[0]
    ref = {"published_P@1": float(row["P@1"]), "published_R@1": float(row["R@1"]), "published_MRR@3": float(row["MRR@3"])}
(OUT / "baseline_protocol.json").write_text(json.dumps({"metrics": {**res, **ref}, "note": "published_* values are constants copied by the original notebook from the paper; not re-run"}, indent=1))
write_manifest("baseline_protocol", "paper-metric-definitions", "avg", {"definitions": "arXiv:2401.10337 App. D.1"}, [p], [OUT / "baseline_protocol.json"], res, "DONE")

## Part 2: why the published model was not re-run, and how to do it

- Their code, weights, predictions and split files are **not** in the project folder. Only two of their data files are (`ttp-desc-mappings.csv` and, by its three source values, `combined_train.csv`).
- Open points that a re-run would settle (see `_evidence_pack/08_baseline.md`): whether their TRAM test split is row-identical to ours; whether their Table 2 was trained in "separate" or "combined" mode.
- The paper states an average training time of 24 hours on one V100 (Limitations section). This is outside a free Colab session.

To run it when compute is available (NOT RUN; the repository layout was not inspected, so the commands after the clone are placeholders to adapt):

```
git clone https://github.com/tumeteor/mitre-ttp-mapping
# 1. compare their TRAM test file with ours:  sha256 of each normalised sentence, then set difference
# 2. train their "Asymmetric" model in SEPARATE mode on TRAM train only
# 3. save its score matrix over the 733 labels as test_scores.npy
# 4. score it with paper_metrics() above and with scripts/verify_claims.py conventions
```

In [ ]:
# ---- evidence table: read back from the files this notebook wrote (no typed numbers) ----
rows = []
for p in sorted(OUT.glob("*.json")):
    d = json.loads(p.read_text())
    flat = d.get("metrics", d) if isinstance(d, dict) else {}
    for k, v in flat.items():
        if isinstance(v, (int, float)) and not isinstance(v, bool): rows.append({"file": p.name, "key": k, "value": round(float(v), 4)})
ev = pd.DataFrame(rows)
print(f"status: {'SMOKE (not a result)' if SMOKE else 'RUN'} | files in {OUT}: {len(list(OUT.glob('*')))}")
print(ev.to_string(index=False) if len(ev) else "no numeric outputs")